# Seminar 5: API (JSON / HTML) - SOLUTIONS
---
Boris Gerát - 10.11

In [ ]:
%pip install pandas requests beautifulsoup4

**TASK** Import the libraries. If you don't have them installed yet, use the code cell above to install them.

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

### PART 0: Warm-up with made-up JSON data

Before we download real data, let's practise on a small example that has the same structure as the World Bank data.

**Reminder:**
- a **list** is indexed with a number: `my_list[0]`
- a **dictionary** is indexed with a key name: `my_dict["value"]`

JSON data is usually a mix of both.

In [ ]:
# made-up numbers, same structure as the World Bank response
toy_data = [
    {"page": 1, "total": 3},
    [
        {"country": {"id": "CZ", "value": "Czechia"}, "date": "2022", "value": 290.0},
        {"country": {"id": "CZ", "value": "Czechia"}, "date": "2021", "value": 282.0},
        {"country": {"id": "CZ", "value": "Czechia"}, "date": "2020", "value": 251.0},
    ],
]

**TASK** Check the type of `toy_data` and how many elements it has.

In [ ]:
print(type(toy_data))
print(len(toy_data))

**TASK** Print the first element of the list. What type is it?

In [ ]:
print(toy_data[0])
print(type(toy_data[0]))

**TASK** The second element is a list of observations. Print the first observation.

In [ ]:
print(toy_data[1][0])

**TASK** Print the value (the number) of the first observation.

*Hint: an observation is a dictionary, so index it with the key name.*

In [ ]:
print(toy_data[1][0]["value"])

**TASK** Print the country name of the first observation.

*Hint: `country` is a dictionary inside a dictionary, so you need two keys.*

In [ ]:
print(toy_data[1][0]["country"]["value"])

**TASK** Loop through all observations and print the year and the value.

*Hint: start with `for obs in toy_data[1]:`*

In [ ]:
for obs in toy_data[1]:
    print(obs["date"], obs["value"])

### PART 1: JSON API

**TASK** Use the provided URL and parameters to load the World Bank data with the `requests` package.

*Hint: use the `get` function.*

In [ ]:
url = "https://api.worldbank.org/v2/country/CZ/indicator/NY.GDP.MKTP.CD"
params = {"format": "json", "date": "2015:2020", "per_page": 100}

In [ ]:
response = requests.get(url, params=params)
print(response.status_code)  # 200 means OK

**TASK** Convert the response to JSON.

*Hint: it is a method of the response object.*

In [ ]:
data = response.json()

**TASK** Check the type of the response object and the type of the converted JSON data.

In [ ]:
print(type(response))
print(type(data))

**TASK** Try indexing the list to get to the data. How deep do you have to go?

In [ ]:
print(data[0])                        # metadata about the request
print(data[1][0])                     # first observation
print(data[1][0]["country"]["value"])  # country name: four levels deep

**TASK** Print how many years there are.

*Hint: first go to the second element of the list.*

In [ ]:
print(len(data[1]))

**TASK** Print the GDP value of the first observed year.

*Note: the API returns the most recent year first.*

In [ ]:
print(data[1][0]["date"], data[1][0]["value"])

**TASK** Print all years and their GDP values.

In [ ]:
for obs in data[1]:
    print(obs["date"], obs["value"])

**TASK** Store all dates, values and countries in a pandas DataFrame.

*Hint: build a list of dictionaries first, then pass it to `pd.DataFrame`.*

In [ ]:
rows = []
for obs in data[1]:
    rows.append({
        "date": obs["date"],
        "value": obs["value"],
        "country": obs["country"]["value"],
    })

df_gdp = pd.DataFrame(rows)
df_gdp

### PART 2: HTML web scraping

**TASK** Use the provided link to load the site data with the `requests` package.

*Hint: use the `get` function.*

In [ ]:
link = "https://www.federalreserve.gov/newsevents/speech/1996speech.htm"

In [ ]:
page = requests.get(link)
print(page.status_code)

**TASK** Try accessing the text of the page from the response.

*Hint: it is an attribute of the response object (no brackets). Print only the first 500 characters.*

In [ ]:
print(page.text[:500])

**TASK** Use BeautifulSoup to turn the raw text into a soup object.

*Hint: pass `"html.parser"` as the second argument.*

In [ ]:
soup = BeautifulSoup(page.text, "html.parser")

**TASK** Use the `prettify()` method on the soup object. What has changed?

*Hint: print only a part of it, e.g. `[:2000]`.*

In [ ]:
print(soup.prettify()[:2000])

**TASK** Find the element that contains all the speeches using the `find` method.

*Hint: inspect the website in your browser.*

In [ ]:
speech_list = soup.find("ul", id="speechIndex")

**TASK** Now find all individual speeches using the `find_all` method.

*Hint: inspect the website in your browser. Each speech is an `li` tag.*

In [ ]:
speeches = speech_list.find_all("li")
print(len(speeches))

**TASK** Try to extract the date of the first speech.

*Hint: the date is the first item of `.contents`. Use `strip()` to remove the extra whitespace.*

In [ ]:
first = speeches[0]
date = first.contents[0].strip()
print(date)

**TASK** Now extract the title of the first speech.

*Hint: use the `find` and `get_text` methods.*

In [ ]:
title = first.find("div", class_="title").get_text(strip=True)
print(title)

**TASK** Now display the title, date, speaker and location of the first speech.

In [ ]:
speaker = first.find("div", class_="speaker").get_text(strip=True)
location = first.find("div", class_="location").get_text(strip=True)

print("Title:   ", title)
print("Date:    ", date)
print("Speaker: ", speaker)
print("Location:", location)

**TASK** Iterate through all speeches and save everything in a pandas DataFrame.

In [ ]:
rows = []
for li in speeches:
    rows.append({
        "title": li.find("div", class_="title").get_text(strip=True),
        "date": li.contents[0].strip(),
        "speaker": li.find("div", class_="speaker").get_text(strip=True),
        "location": li.find("div", class_="location").get_text(strip=True),
    })

df_speeches = pd.DataFrame(rows)
df_speeches